# Finance questions on Kaggle (after the build)

Add the work zip as an input, set the accelerator to GPU T4, then Save Version > Save & Run All. Questions stop by `HOURS` after the set-up; the report gets 20 minutes after that.

In [ ]:
# 1 Setup. The work folder comes in as a Kaggle dataset: Add Input > your dataset with
# work_finance_bank_high.zip (made on the laptop after the build). No Gemini key is needed.
SOURCE = "bank"
SCALE = "high"
HOURS = 2.5          # questions stop 10 min before this; then the report has its own 20 min
MODEL = "llama3:8b"
FREEZE = "eval-freeze-28"

import glob, os, subprocess, sys, time, zipfile
from pathlib import Path
START = time.time()
W = Path("/kaggle/working")
REPO = W / "3rd_sem_el"
NAME = f"finance_{SOURCE}_{SCALE}"
WORK = W / "eval" / NAME
import shutil
found = glob.glob(f"/kaggle/input/**/work_{NAME}.zip", recursive=True)
unzipped = [p for p in glob.glob(f"/kaggle/input/**/{NAME}", recursive=True) if Path(p, "stores").is_dir()]
if not WORK.exists():
    if found:
        with zipfile.ZipFile(found[0]) as z:
            z.extractall(WORK.parent)
        print("work folder restored from", found[0])
    elif unzipped:  # Kaggle unzips an uploaded zip into a folder
        shutil.copytree(unzipped[0], WORK)
        print("work folder copied from", unzipped[0])
    else:
        raise RuntimeError(f"add the dataset with work_{NAME}.zip as an input (Add Input), then run again")
print("stores:", sorted(p.name for p in (WORK / "stores").glob("*.sqlite")))
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", FREEZE,
                    "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"
!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5" matplotlib
sys.path.insert(0, str(REPO / "tools"))
import colab_cache
colab_cache.install_ollama_and_model(MODEL)
colab_cache.start_ollama()
from huggingface_hub import snapshot_download
for attempt in range(6):  # the search model, before any question needs it
    try:
        snapshot_download("BAAI/bge-m3"); break
    except Exception as exc:
        print(f"embedding model download failed ({exc}); again in {20 * 2 ** attempt} s")
        time.sleep(20 * 2 ** attempt)
os.environ.update({
    "OLLAMA_NUM_PARALLEL": "4", "EVAL_CONCURRENCY": "2",
    "SANITISE_LLM_BACKEND": "ollama", "SANITISE_MODEL": MODEL, "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid",
    "SANITISE_MONITOR_DB_PATH": str(WORK / "stores" / "monitor.sqlite"),
})
DEADLINE = time.time() + HOURS * 3600   # counted from here, after the set-up
os.environ["EVAL_DEADLINE"] = str(DEADLINE)
print(f"set-up took {(time.time() - START) / 60:.0f} min; questions stop by "
      f"{time.strftime('%H:%M', time.gmtime(DEADLINE - 600))} UTC, report by "
      f"{time.strftime('%H:%M', time.gmtime(DEADLINE + 1200))} UTC")
RUN = f"python tools/eval_suite/run.py {{step}} --domain finance --source {SOURCE} --work {WORK} --scale {SCALE}"
def run(step, limit_s):
    colab_cache.start_ollama()
    cmd = RUN.format(step=step)
    print("$", cmd, f"(at most {limit_s / 60:.0f} min)", flush=True)
    (WORK / "results").mkdir(parents=True, exist_ok=True)
    try:  # a hard stop: the step is ended at its limit whatever it is doing
        subprocess.run(["bash", "-o", "pipefail", "-c",
                        "(" + cmd + ") 2>&1 | tee -a '" + str(WORK / "results" / "run.log") + "'"],
                       check=False, timeout=max(60, limit_s), env={**os.environ, "PYTHONUNBUFFERED": "1"})
    except subprocess.TimeoutExpired:
        print(f"!!! {step} stopped at its time limit; what it finished is kept", flush=True)


In [ ]:
# 2 Questions and report. Every step resumes and stops on time; the report always runs.
for step in ["single", "multi", "e2e"]:
    run(step, DEADLINE - time.time())
run("report", 20 * 60)


In [ ]:
# 3 Pack the results (Output tab)
zip_path = W / f"eval_results_{NAME}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/utility.json", "pii/labels.json", "e2e/labels.json",
        "e2e/manifest.json", "stores/build.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"{NAME}/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB; whole session {(time.time() - START) / 3600:.2f} h")
